# Bootstrap secrets
#
# Run by hand, once per environment. Creates the `finhive` Databricks secret scope
# and populates every secret the code references by name. Values are typed in
# interactively (via widgets or `getpass`) - never hardcoded in this notebook.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists
import getpass

dbutils.widgets.text("scope", "finhive")
scope = dbutils.widgets.get("scope")

# Secret names the pipelines need. Names are not secret - only the values are -
# so this list is checked in and reviewed like any other code (ARCHITECTURE_V2.md #5.2).
REQUIRED_KEYS = [
    "fred_api_key",
]

w = WorkspaceClient()

In [ ]:
def sanitize(value: str) -> str:
    """Strip whitespace and a leading UTF-8 BOM (the Windows copy/paste bug)."""
    return value.strip().lstrip("\ufeff")


try:
    w.secrets.create_scope(scope)
except ResourceAlreadyExists:
    pass

for key in REQUIRED_KEYS:
    raw_value = dbutils.widgets.get(key) if key in dbutils.widgets.getAll() else ""
    if not raw_value:
        raw_value = getpass.getpass(f"{key}: ")

    value = sanitize(raw_value)
    if not value:
        raise ValueError(f"no value provided for required secret '{key}'")

    w.secrets.put_secret(scope=scope, key=key, string_value=value)
    print(f"stored secret '{key}' in scope '{scope}'")